In [1]:
pip install hdfs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 610.5 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for hdfs: filename=hdfs-2.7.3-py3-none-any.whl size=34325 sha256=1332f32cb340cac412373807753afd3b620c1cd63dc2e87229618b714ba6bce9
  Stored in directory: /home/jovyan/.cache/pip/wheels/b9/1d/dc/eb0833be25464c359903d356c4204721c6a672c26ff164cdc3
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13705 sha256=5da972ab85d65a0c683464d2927e1775260877e48705280ca52e666dc4f3a2be
  Stored in directory: /home/jovyan/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built hdfs docopt
Note: you may need to restart the kernel to use updated packages.


In [1]:
from hdfs import InsecureClient
import os

# Create a HDFS connector client
hdfs_client = InsecureClient("http://hive:50070", user='root')

# List HDFS file and directories
print(hdfs_client.list('/user/gravitino'))

hdfs_client.delete("/user/gravitino")

['schema']


HdfsError: `/user/gravitino is non empty': Directory is not empty

In [1]:
pip install apache-gravitino==0.9.1

  Using cached apache_gravitino-0.9.1.tar.gz (89 kB)
  Preparing metadata (setup.py) ... done
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached readerwriterlock-1.0.9-py3-none-any.whl.metadata (5.6 kB)
  Using cached fsspec-2024.3.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached pyarrow-15.0.2-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached gcsfs-2024.3.1-py2.py3-none-any.whl.metadata (1.6 kB)
  Using cached s3fs-2024.3.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached ossfs-2023.12.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached adlfs-2023.12.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached azure_core-1.35.0-py3-none-any.whl.metadata (44 kB)
  Using cached azure_datalake_store-0.0.53-py2.py3-none-any.whl.metadata (19 kB)
  Using cached azure_identity-1.24.0-py3-none-any.whl.metadata (86 kB)
  Using cached azure_storage_blob-12.26.0-py3-none-any.whl

In [2]:
from typing import Dict, List
from gravitino import NameIdentifier, GravitinoAdminClient, GravitinoClient, Catalog, Fileset, FilesetChange
import os 

# Create Gravitino admin client
gravitino_admin_client = GravitinoAdminClient(uri="http://gravitino:8090")

# Create metalake via Gravitino admin client
metalake_name="default"
metalake = gravitino_admin_client.create_metalake(name=metalake_name,
                                                  comment="metalake comment", 
                                                  properties={})
print(metalake)

GravitinoMetalake(_name='default', _comment='metalake comment', _properties={}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-08-28T04:01:56.061366527Z', _last_modifier=None, _last_modified_time=None))


In [3]:
# Create Gravitino client
gravitino_client = GravitinoClient(uri="http://gravitino:8090", metalake_name=metalake_name)

In [4]:
from typing import Dict, List
from gravitino import GravitinoMetalake

# List all Gravitino metalake entity
metalake_list: List[GravitinoMetalake] = gravitino_admin_client.list_metalakes()
print(metalake_list)

[GravitinoMetalake(_name='default', _comment='metalake comment', _properties={'in-use': 'true'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-08-28T04:01:56.061366527Z', _last_modifier=None, _last_modified_time=None)), GravitinoMetalake(_name='metalake_demo', _comment='comment', _properties={'in-use': 'true'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-08-28T02:26:31.903813324Z', _last_modifier=None, _last_modified_time=None))]


In [5]:
# Create catalog via Gravition client
catalog_name="catalog"

catalog = gravitino_client.create_catalog(name=catalog_name,
                                          catalog_type=Catalog.Type.FILESET,
                                          provider="hadoop", 
                                          comment="",
                                          properties={})
print(catalog)

FilesetCatalog(_name='catalog', _type=<Type.FILESET: ('fileset', False)>, _provider='hadoop', _comment='', _properties={'in-use': 'true'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-08-28T04:02:32.905432713Z', _last_modifier='anonymous', _last_modified_time='2025-08-28T04:02:32.905432713Z'))


In [6]:
# Load catalog entity via Gravition client
catalog = gravitino_client.load_catalog(name=catalog_name)
print(catalog)

FilesetCatalog(_name='catalog', _type=<Type.FILESET: ('fileset', False)>, _provider='hadoop', _comment='', _properties={'in-use': 'true'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-08-28T04:02:32.905432713Z', _last_modifier='anonymous', _last_modified_time='2025-08-28T04:02:32.905432713Z'))


In [7]:
# Create schema entity via Gravition client
schema_name="schema"
schema_path="/user/gravitino/"+schema_name
schema_hdfs_path=f"hdfs://hive:9000{schema_path}"

catalog.as_schemas().create_schema(schema_name=schema_name, 
                                   comment="", 
                                   properties={"location":schema_hdfs_path})

# Check if the schema location was successfully created in HDFS
try:
    info = hdfs_client.status(schema_path)
    print(f"Success: The storage location {schema_path} was successfully created.")
    print("Details:", info)
except Exception:
    print(f"Faild: The storage location {schema_path} was not successfully created.")

INFO:hdfs.client:Fetching status for '/user/gravitino/schema'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema' to '/user/gravitino/schema'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema?user.name=root&op=GETFILESTATUS HTTP/1.1" 200 None


Success: The storage location /user/gravitino/schema was successfully created.
Details: {'accessTime': 0, 'blockSize': 0, 'childrenNum': 1, 'fileId': 17429, 'group': 'hdfs', 'length': 0, 'modificationTime': 1756352229131, 'owner': 'root', 'pathSuffix': '', 'permission': '755', 'replication': 0, 'storagePolicy': 0, 'type': 'DIRECTORY'}


In [8]:

# Create a managed type of Fileset
managed_fileset_name="managed_fileset"
managed_fileset_path="/user/gravitino/"+schema_name+"/"+managed_fileset_name
managed_fileset_hdfs_path=f"hdfs://hive:9000{managed_fileset_path}"

managed_fileset_ident: NameIdentifier = NameIdentifier.of(schema_name, managed_fileset_name)
catalog.as_fileset_catalog().create_fileset(ident=managed_fileset_ident,
                                            fileset_type=Fileset.Type.MANAGED,
                                            comment="",
                                            storage_location=managed_fileset_hdfs_path,
                                            properties={})

# Check if the fileset location was successfully created in HDFS
try:
    info = hdfs_client.status(managed_fileset_path)
    print(f"Success: The storage location {managed_fileset_path} was successfully created.")
    print("Details:", info)  # print hdfs path detail informations
except Exception:
    print(f"Faild: The storage location {managed_fileset_path} was not successfully created.")

INFO:hdfs.client:Fetching status for '/user/gravitino/schema/managed_fileset'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema/managed_fileset' to '/user/gravitino/schema/managed_fileset'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema/managed_fileset?user.name=root&op=GETFILESTATUS HTTP/1.1" 200 None


Success: The storage location /user/gravitino/schema/managed_fileset was successfully created.
Details: {'accessTime': 0, 'blockSize': 0, 'childrenNum': 0, 'fileId': 17545, 'group': 'hdfs', 'length': 0, 'modificationTime': 1756353801868, 'owner': 'root', 'pathSuffix': '', 'permission': '755', 'replication': 0, 'storagePolicy': 0, 'type': 'DIRECTORY'}


In [9]:
external_fileset_name="external_fileset"
external_fileset_path="/user/gravitino/"+schema_name+"/"+external_fileset_name
external_fileset_hdfs_path=f"hdfs://hive:9000{external_fileset_path}"

# Create a fileset path in HDFS in advance
hdfs_client.makedirs(external_fileset_path)
try:
    info = hdfs_client.status(external_fileset_path)
    print(f"Success: The storage location {external_fileset_path} was successfully created.")
    print("Details:", info)  # print hdfs path detail information
except Exception:
    print(f"Faild: The storage location {external_fileset_path} was not successfully created.")

# Create an external type of fileset
external_fileset_ident: NameIdentifier = NameIdentifier.of(schema_name, external_fileset_name)
catalog.as_fileset_catalog().create_fileset(ident=external_fileset_ident,
                                            fileset_type=Fileset.Type.EXTERNAL,
                                            comment="",
                                            storage_location=external_fileset_hdfs_path,
                                            properties={})

INFO:hdfs.client:Creating directories to '/user/gravitino/schema/external_fileset'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema/external_fileset' to '/user/gravitino/schema/external_fileset'.
DEBUG:urllib3.connectionpool:http://hive:50070 "PUT /webhdfs/v1/user/gravitino/schema/external_fileset?user.name=root&op=MKDIRS HTTP/1.1" 200 None
INFO:hdfs.client:Fetching status for '/user/gravitino/schema/external_fileset'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema/external_fileset' to '/user/gravitino/schema/external_fileset'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema/external_fileset?user.name=root&op=GETFILESTATUS HTTP/1.1" 200 None


Success: The storage location /user/gravitino/schema/external_fileset was successfully created.
Details: {'accessTime': 0, 'blockSize': 0, 'childrenNum': 0, 'fileId': 17433, 'group': 'hdfs', 'length': 0, 'modificationTime': 1756352222294, 'owner': 'root', 'pathSuffix': '', 'permission': '755', 'replication': 0, 'storagePolicy': 0, 'type': 'DIRECTORY'}


In [10]:
# List all fileset
catalog = gravitino_client.load_catalog(name=catalog_name)
fileset_list: List[NameIdentifier] = catalog.as_fileset_catalog().list_filesets(namespace=managed_fileset_ident.namespace())
print(fileset_list)

[NameIdentifier(_name='external_fileset', _namespace=<gravitino.namespace.Namespace object at 0x7f2a6b75ab90>), NameIdentifier(_name='managed_fileset', _namespace=<gravitino.namespace.Namespace object at 0x7f2a6bd85d90>)]


In [11]:
# Load managed fileset
managed_fileset=gravitino_client.load_catalog(name=catalog_name).as_fileset_catalog().load_fileset(ident=managed_fileset_ident)
print(managed_fileset)

In [12]:
# Load external fileset
external_fileset=gravitino_client.load_catalog(name=catalog_name).as_fileset_catalog().load_fileset(ident=external_fileset_ident)
print(external_fileset)

In [13]:
# Drop managed type of fileset and deleted HDFS location
catalog.as_fileset_catalog().drop_fileset(ident=managed_fileset_ident)

# Check managed type of fileset location if successfully deleted
try:
    info = hdfs_client.status(managed_fileset_path)
    print(f"Faild: The storage location {managed_fileset_path} was not successfully deleted.")
except Exception:
    print(f"Success: The storage location {managed_fileset_path} was successfully deleted.")

INFO:hdfs.client:Fetching status for '/user/gravitino/schema/managed_fileset'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema/managed_fileset' to '/user/gravitino/schema/managed_fileset'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema/managed_fileset?user.name=root&op=GETFILESTATUS HTTP/1.1" 404 None


Success: The storage location /user/gravitino/schema/managed_fileset was successfully deleted.


In [14]:
# Drop external type of fileset, Should not be deleted HDFS location
catalog.as_fileset_catalog().drop_fileset(ident=external_fileset_ident)

# Check managed type of fileset location if reserved
try:
    info = hdfs_client.status(external_fileset_path)
    print(f"Success: The storage location {external_fileset_path} reserved.")
except Exception:
    print(f"Faild: The storage location {external_fileset_path} deleted.")

INFO:hdfs.client:Fetching status for '/user/gravitino/schema/external_fileset'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema/external_fileset' to '/user/gravitino/schema/external_fileset'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema/external_fileset?user.name=root&op=GETFILESTATUS HTTP/1.1" 200 None


Success: The storage location /user/gravitino/schema/external_fileset reserved.


In [15]:
# Drop schema
catalog.as_schemas().drop_schema(schema_name=schema_name, cascade=True)

# Check schema location if successfully deleted
try:
    info = hdfs_client.status(schema_path)
    print(f"Faild: The storage location {schema_path} was not successfully deleted.")
except Exception:
    print(f"Success: The storage location {schema_path} was successfully deleted.")

INFO:hdfs.client:Fetching status for '/user/gravitino/schema'.
DEBUG:hdfs.client:Resolved path '/user/gravitino/schema' to '/user/gravitino/schema'.
DEBUG:urllib3.connectionpool:http://hive:50070 "GET /webhdfs/v1/user/gravitino/schema?user.name=root&op=GETFILESTATUS HTTP/1.1" 200 None


Faild: The storage location /user/gravitino/schema was not successfully deleted.


In [16]:
# Drop catalog
result=gravitino_client.drop_catalog(name=catalog_name, force=True)
print(result)

True


In [17]:
# Drop metalake
result=gravitino_admin_client.drop_metalake(metalake_name, force=True)
print(result)

True


In [ ]:
import os
import signal
import ipykernel

# Lấy PID của kernel hiện tại
pid = os.getpid()

print(f"Stopping Jupyter kernel {pid} ...")
os.kill(pid, signal.SIGTERM)